In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
%run ../0-common/bronze_helpers

In [0]:
source_file = f"{landing_folfer_path}/{v_batch_id}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

constructor_schema = StructType([
    StructField("constructorId", StringType()),
    StructField("name", StringType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
constructor_df = (
    spark.read.format('json')
    .option('header', True)
    .schema(constructor_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
)

In [0]:
constructor_df_final = add_ingestion_metadata(constructor_df)

In [0]:
constructor_df_final = constructor_df_final.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
(
    constructor_df_final.write
    .format('delta')
    .mode('overwrite')
    .partitionBy('batch_id')
    .option('replaceWhere', f"batch_id='{v_batch_id}'")
    .saveAsTable(table_name)
)